# 02 数据分析与可视化

本 Notebook 基于清洗后的 Parquet 数据完成探索性数据分析（EDA），覆盖交易时间趋势、商品长尾、类目结构、价格分布、用户活跃度、购物篮、复购间隔、用户画像和冷启动覆盖率。输出文件保存在 `optimization/output/eda/`，可直接用于课程报告。

为保证本地与 Kaggle 均可运行：

- 优先读取本地清洗产物；
- 若在 Kaggle 中已将清洗产物作为附件，则自动搜索 Kaggle 输入目录；
- 若未附加清洗产物，则回退读取原始竞赛数据并执行最小化规范化。

本机未安装中文字体，因此图表标题与坐标轴使用英文，正文分析和结论使用中文。

In [ ]:
import json
import os
import shutil
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-hm-recsys')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', context='notebook')
PALETTE = sns.color_palette('Set2')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 160
plt.rcParams['savefig.bbox'] = 'tight'


def find_clean_dir():
    local = Path.cwd() / 'output' / 'clean'
    candidates = [
        local,
        Path.cwd() / 'optimization' / 'output' / 'clean',
        Path.cwd().parent / 'output' / 'clean',
        Path.cwd().parent / 'optimization' / 'output' / 'clean',
        Path('/kaggle/input/hm-recsys-clean'),
        Path('/kaggle/input/hm_recsys_clean'),
        Path('/kaggle/input/hm-recsys-clean/hm_recsys_clean'),
    ]
    required = {'transactions_clean.parquet', 'customers_clean.parquet', 'articles_clean.parquet', 'submission_customers.parquet'}
    for candidate in candidates:
        if candidate.is_dir() and required.issubset({path.name for path in candidate.iterdir()}):
            return candidate.resolve()
    return None


def find_raw_dir():
    candidates = [
        Path.cwd() / 'data',
        Path.cwd().parent / 'data',
        Path.cwd().parents[1] / 'data',
        Path('/kaggle/input/h-and-m-personalized-fashion-recommendations'),
    ]
    required = {'transactions_train.csv', 'customers.csv', 'articles.csv', 'sample_submission.csv'}
    for candidate in candidates:
        if candidate.is_dir() and required.issubset({path.name for path in candidate.iterdir()}):
            return candidate.resolve()
    raise FileNotFoundError(f'未找到清洗产物或原始数据: {sorted(required)}')


def read_raw_fallback(raw_dir):
    print('未找到清洗产物，回退读取原始数据并执行最小化规范化...')
    with (raw_dir / 'articles.csv').open('rb') as article_file:
        compression = 'zip' if article_file.read(4) == b'PK\x03\x04' else None
    transactions = pd.read_csv(raw_dir / 'transactions_train.csv', dtype={'customer_id': 'string', 'article_id': 'string'})
    transactions['t_dat'] = pd.to_datetime(transactions['t_dat'], format='%Y-%m-%d')
    transactions['customer_key'] = np.fromiter((int(value[-16:], 16) for value in transactions['customer_id']), dtype=np.uint64, count=len(transactions))
    transactions['article_id'] = transactions['article_id'].str.zfill(10)
    transactions['price'] = pd.to_numeric(transactions['price'], errors='raise')
    transactions['sales_channel_id'] = pd.to_numeric(transactions['sales_channel_id'], errors='raise').astype('int8')
    transactions = transactions[['t_dat', 'customer_key', 'article_id', 'price', 'sales_channel_id']]
    customers = pd.read_csv(raw_dir / 'customers.csv', dtype={'customer_id': 'string'})
    customers['customer_key'] = np.fromiter((int(value[-16:], 16) for value in customers['customer_id']), dtype=np.uint64, count=len(customers))
    customers = customers.rename(columns={'FN': 'fn_flag', 'Active': 'active_flag'})
    articles = pd.read_csv(raw_dir / 'articles.csv', compression=compression, dtype={'article_id': 'string', 'product_code': 'int32'})
    articles['article_id'] = articles['article_id'].str.zfill(10)
    submission = pd.read_csv(raw_dir / 'sample_submission.csv', usecols=['customer_id'], dtype={'customer_id': 'string'})
    submission['customer_key'] = np.fromiter((int(value[-16:], 16) for value in submission['customer_id']), dtype=np.uint64, count=len(submission))
    return transactions, customers, articles, submission


CLEAN_DIR = find_clean_dir()
RAW_DIR = None if CLEAN_DIR else find_raw_dir()
if CLEAN_DIR:
    transactions = pd.read_parquet(CLEAN_DIR / 'transactions_clean.parquet')
    customers = pd.read_parquet(CLEAN_DIR / 'customers_clean.parquet')
    articles = pd.read_parquet(CLEAN_DIR / 'articles_clean.parquet')
    submission = pd.read_parquet(CLEAN_DIR / 'submission_customers.parquet')
else:
    transactions, customers, articles, submission = read_raw_fallback(RAW_DIR)

transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])
LOCAL_PROJECT_DIR = next((base / 'optimization' for base in (Path.cwd(), *Path.cwd().parents) if (base / 'optimization' / 'code').is_dir()), Path.cwd())
EDA_DIR = Path('/kaggle/working/hm_recsys_eda') if Path('/kaggle/working').exists() else LOCAL_PROJECT_DIR / 'output' / 'eda'
FIGURE_DIR = EDA_DIR / 'figures'
TABLE_DIR = EDA_DIR / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

transactions['article_num'] = transactions['article_id'].astype('int32')
transactions['year_month'] = transactions['t_dat'].dt.to_period('M').astype(str)


def save_figure(fig, name):
    path = FIGURE_DIR / f'{name}.png'
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)
    print('saved:', path)
    return path


def save_table(frame, name):
    path = TABLE_DIR / f'{name}.csv'
    frame.to_csv(path, index=False)
    print('saved:', path)
    return path


def comma_formatter(value, _):
    return f'{value:,.0f}'

print('CLEAN_DIR:', CLEAN_DIR)
print('RAW_DIR:', RAW_DIR)
print('transactions:', transactions.shape)
print('customers:', customers.shape)
print('articles:', articles.shape)
print('submission:', submission.shape)
display(transactions.head())

## 1. 交易时间趋势

分析每日交易量、销售额、7 日平滑趋势、月度渠道结构与星期效应，用于识别季节性、促销窗口和线上线下行为差异。

In [ ]:
temporal_started = time.time()
daily = transactions.groupby('t_dat', as_index=False).agg(
    purchases=('article_num', 'size'),
    revenue=('price', 'sum'),
    active_customers=('customer_key', 'nunique'),
    active_articles=('article_num', 'nunique'),
)
daily['purchases_7d'] = daily['purchases'].rolling(7, min_periods=1).mean()
daily['revenue_7d'] = daily['revenue'].rolling(7, min_periods=1).mean()
save_table(daily, 'daily_transaction_summary')

fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
axes[0].plot(daily['t_dat'], daily['purchases'], color='#9ecae1', linewidth=0.8, alpha=0.55, label='Daily purchases')
axes[0].plot(daily['t_dat'], daily['purchases_7d'], color='#08519c', linewidth=2.0, label='7-day average')
axes[0].set_ylabel('Purchases')
axes[0].set_title('Daily purchase trend')
axes[0].legend()
axes[1].plot(daily['t_dat'], daily['revenue'], color='#fdd0a2', linewidth=0.8, alpha=0.55, label='Daily revenue')
axes[1].plot(daily['t_dat'], daily['revenue_7d'], color='#e6550d', linewidth=2.0, label='7-day average')
axes[1].set_ylabel('Revenue')
axes[1].set_xlabel('Date')
axes[1].set_title('Daily revenue trend')
axes[1].legend()
for axis in axes:
    axis.yaxis.set_major_formatter(FuncFormatter(comma_formatter))
fig.tight_layout()
save_figure(fig, '01_daily_trend')

monthly_channel = transactions.pivot_table(
    index='year_month', columns='sales_channel_id', values='price', aggfunc='size', fill_value=0
).rename(columns=lambda value: f'Channel {value}')
monthly_channel_ratio = monthly_channel.div(monthly_channel.sum(axis=1), axis=0)
save_table(monthly_channel, 'monthly_channel_purchases')
save_table(monthly_channel_ratio, 'monthly_channel_ratio')
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
monthly_channel.plot.area(ax=axes[0], color=PALETTE[:2], alpha=0.85)
axes[0].set_title('Monthly purchases by sales channel')
axes[0].set_ylabel('Purchases')
axes[0].yaxis.set_major_formatter(FuncFormatter(comma_formatter))
monthly_channel_ratio.plot.area(ax=axes[1], color=PALETTE[:2], alpha=0.85, ylim=(0, 1))
axes[1].set_title('Monthly channel share')
axes[1].set_ylabel('Share')
axes[1].set_xlabel('Month')
axes[1].tick_params(axis='x', rotation=45)
fig.tight_layout()
save_figure(fig, '02_monthly_channel_trend')

transactions['day_name'] = transactions['t_dat'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = transactions.groupby('day_name').agg(purchases=('article_num', 'size'), revenue=('price', 'sum')).reindex(dow_order)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=dow.index, y=dow['purchases'], hue=dow.index, ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Purchases by day of week')
axes[0].set_xlabel('Day of week')
axes[0].set_ylabel('Purchases')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(FuncFormatter(comma_formatter))
sns.barplot(x=dow.index, y=dow['revenue'], hue=dow.index, ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('Revenue by day of week')
axes[1].set_xlabel('Day of week')
axes[1].set_ylabel('Revenue')
axes[1].tick_params(axis='x', rotation=30)
fig.tight_layout()
save_figure(fig, '03_day_of_week_pattern')

temporal_summary = {
    'date_min': str(daily['t_dat'].min().date()),
    'date_max': str(daily['t_dat'].max().date()),
    'daily_purchase_mean': float(daily['purchases'].mean()),
    'daily_purchase_max': int(daily['purchases'].max()),
    'daily_purchase_max_date': str(daily.loc[daily['purchases'].idxmax(), 't_dat'].date()),
    'daily_revenue_mean': float(daily['revenue'].mean()),
    'channel_purchase_share': (transactions['sales_channel_id'].value_counts(normalize=True).sort_index() * 100).round(2).to_dict(),
    'elapsed_seconds': round(time.time() - temporal_started, 3),
}
print(json.dumps(temporal_summary, ensure_ascii=False, indent=2))

## 2. 商品结构与长尾分布

分析商品销量、销售额、类目、价格带与热门集中度。长尾分布直接决定全局热门召回的适用边界，也说明个性化召回与冷启动策略的必要性。

In [ ]:
item_started = time.time()
item_stats = transactions.groupby('article_num', as_index=False).agg(
    purchases=('article_num', 'size'),
    revenue=('price', 'sum'),
    customers=('customer_key', 'nunique'),
    first_date=('t_dat', 'min'),
    last_date=('t_dat', 'max'),
)
article_columns = [
    'article_id', 'article_num', 'prod_name', 'product_type_name', 'product_group_name',
    'index_group_name', 'garment_group_name', 'colour_group_name', 'department_name'
]
articles_work = articles.copy()
articles_work['article_num'] = articles_work['article_id'].astype('int32')
item_stats = item_stats.merge(articles_work[article_columns], on='article_num', how='left')
item_stats = item_stats.sort_values('purchases', ascending=False).reset_index(drop=True)
item_stats['rank'] = np.arange(1, len(item_stats) + 1)
item_stats['cum_purchase_share'] = item_stats['purchases'].cumsum() / item_stats['purchases'].sum()
save_table(item_stats.head(500), 'top500_articles')

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot(item_stats['rank'], item_stats['purchases'], color='#08519c', linewidth=1.3)
axes[0].set_title('Article popularity long-tail curve')
axes[0].set_xlabel('Article popularity rank')
axes[0].set_ylabel('Purchases')
axes[1].plot(item_stats['rank'], item_stats['cum_purchase_share'], color='#e6550d', linewidth=1.8)
for top_n, color in [(10, '#31a354'), (100, '#756bb1'), (1000, '#d95f0e')]:
    share = item_stats.loc[top_n - 1, 'cum_purchase_share']
    axes[1].axvline(top_n, color=color, linestyle='--', linewidth=1.1)
    axes[1].annotate(f'Top {top_n}: {share:.1%}', xy=(top_n, share), xytext=(top_n + 3500, share - 0.08), color=color)
axes[1].set_title('Cumulative purchase coverage by article rank')
axes[1].set_xlabel('Top-N articles')
axes[1].set_ylabel('Cumulative purchase share')
axes[1].set_xlim(1, len(item_stats))
axes[1].set_ylim(0, 1)
fig.tight_layout()
save_figure(fig, '04_article_long_tail')

def plot_top_category(column, title, name, top_n=15):
    summary = item_stats.groupby(column, observed=True).agg(purchases=('purchases', 'sum'), revenue=('revenue', 'sum'), articles=('article_num', 'nunique')).sort_values('purchases', ascending=False).head(top_n)
    fig, axis = plt.subplots(figsize=(12, max(5, 0.45 * len(summary))))
    # Use matplotlib directly: passing a pandas Series together with a categorical hue
    # can produce an empty bar layer under some seaborn versions.
    axis.barh(summary.index.astype(str), summary['purchases'].to_numpy(), color='#66c2a5')
    axis.set_title(title)
    axis.set_xlabel('Purchases')
    axis.set_ylabel('')
    axis.xaxis.set_major_formatter(FuncFormatter(comma_formatter))
    fig.tight_layout()
    save_figure(fig, name)
    return summary

product_group_summary = plot_top_category('product_group_name', 'Purchases by product group', '05_product_group_purchases')
index_group_summary = plot_top_category('index_group_name', 'Purchases by index group', '06_index_group_purchases')
color_summary = plot_top_category('colour_group_name', 'Purchases by colour group', '07_colour_group_purchases')
save_table(product_group_summary.reset_index(), 'product_group_summary')
save_table(index_group_summary.reset_index(), 'index_group_summary')
save_table(color_summary.reset_index(), 'colour_group_summary')

channel_group = transactions.merge(articles_work[['article_num', 'product_group_name']], on='article_num', how='left').groupby(['product_group_name', 'sales_channel_id'], observed=True).size().unstack(fill_value=0)
top_groups = channel_group.sum(axis=1).sort_values(ascending=False).head(12).index
channel_group_ratio = channel_group.loc[top_groups].div(channel_group.loc[top_groups].sum(axis=1), axis=0)
fig, axis = plt.subplots(figsize=(13, 6.5))
channel_group_ratio.plot.barh(stacked=True, ax=axis, color=PALETTE[:2], width=0.78)
axis.set_title('Sales channel share by top product groups')
axis.set_xlabel('Channel share')
axis.set_ylabel('')
axis.set_xlim(0, 1)
axis.legend(title='Sales channel', loc='lower right')
fig.tight_layout()
save_figure(fig, '08_product_group_channel_share')

rng = np.random.default_rng(2026)
price_sample = transactions.sample(min(1_500_000, len(transactions)), random_state=2026)[['price', 'sales_channel_id']].copy()
price_sample['channel'] = price_sample['sales_channel_id'].map({1: 'Channel 1', 2: 'Channel 2'})
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.histplot(price_sample['price'], bins=60, kde=True, ax=axes[0], color='#08519c')
axes[0].set_title('Transaction price distribution')
axes[0].set_xlabel('Price')
axes[0].set_ylabel('Sampled purchases')
sns.boxplot(data=price_sample, x='channel', y='price', hue='channel', ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('Price distribution by sales channel')
axes[1].set_xlabel('Sales channel')
axes[1].set_ylabel('Price')
fig.tight_layout()
save_figure(fig, '09_price_distribution')

item_summary = {
    'active_articles': int(item_stats['article_num'].nunique()),
    'top10_purchase_share': float(item_stats.loc[9, 'cum_purchase_share']),
    'top100_purchase_share': float(item_stats.loc[99, 'cum_purchase_share']),
    'top1000_purchase_share': float(item_stats.loc[999, 'cum_purchase_share']),
    'median_article_purchases': float(item_stats['purchases'].median()),
    'price_mean': float(transactions['price'].mean()),
    'price_median': float(transactions['price'].median()),
    'price_p05': float(transactions['price'].quantile(0.05)),
    'price_p95': float(transactions['price'].quantile(0.95)),
    'elapsed_seconds': round(time.time() - item_started, 3),
}
print(json.dumps(item_summary, ensure_ascii=False, indent=2))

## 3. 用户行为、购物篮与复购

从交易、活跃天、购物篮大小、访问间隔和同商品复购间隔刻画用户行为。这些统计将为时间衰减复购、多粒度共现和长期偏好召回提供依据。

In [ ]:
behavior_started = time.time()
customer_stats = transactions.groupby('customer_key', as_index=False).agg(
    purchases=('article_num', 'size'),
    revenue=('price', 'sum'),
    active_days=('t_dat', 'nunique'),
    unique_articles=('article_num', 'nunique'),
    first_date=('t_dat', 'min'),
    last_date=('t_dat', 'max'),
)
customer_stats['recency_days'] = (customer_stats['last_date'].max() - customer_stats['last_date']).dt.days
customer_stats['avg_basket_value'] = customer_stats['revenue'] / customer_stats['active_days']
customer_stats['avg_items_per_active_day'] = customer_stats['purchases'] / customer_stats['active_days']
customer_stats['active_span_days'] = (customer_stats['last_date'] - customer_stats['first_date']).dt.days
save_table(customer_stats, 'customer_transaction_stats')

activity_bins = [0, 1, 4, 9, 24, 99, np.inf]
activity_labels = ['1', '2-4', '5-9', '10-24', '25-99', '100+']
customer_stats['purchase_segment'] = pd.cut(customer_stats['purchases'], bins=activity_bins, labels=activity_labels, right=True)
activity_summary = customer_stats.groupby('purchase_segment', observed=False).agg(
    customers=('customer_key', 'size'), purchases=('purchases', 'sum'), revenue=('revenue', 'sum')
).reset_index()
activity_summary['customer_share'] = activity_summary['customers'] / activity_summary['customers'].sum()
activity_summary['purchase_share'] = activity_summary['purchases'] / activity_summary['purchases'].sum()
save_table(activity_summary, 'customer_activity_summary')

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
sns.barplot(data=activity_summary, x='purchase_segment', y='customer_share', hue='purchase_segment', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Customer share by lifetime purchases')
axes[0].set_xlabel('Lifetime purchases')
axes[0].set_ylabel('Customer share')
axes[0].tick_params(axis='x', rotation=20)
sns.barplot(data=activity_summary, x='purchase_segment', y='purchase_share', hue='purchase_segment', ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('Purchase share by customer activity')
axes[1].set_xlabel('Lifetime purchases')
axes[1].set_ylabel('Purchase share')
axes[1].tick_params(axis='x', rotation=20)
sns.histplot(customer_stats.loc[customer_stats['purchases'] <= 100, 'purchases'], bins=50, ax=axes[2], color='#08519c')
axes[2].set_title('Customer purchase count distribution')
axes[2].set_xlabel('Purchases (clipped at 100)')
axes[2].set_ylabel('Customers')
fig.tight_layout()
save_figure(fig, '10_customer_activity_distribution')

basket_stats = transactions.groupby(['customer_key', 't_dat'], as_index=False).agg(
    basket_items=('article_num', 'size'), unique_articles=('article_num', 'nunique'), basket_value=('price', 'sum')
)
basket_stats['basket_items_clipped'] = basket_stats['basket_items'].clip(upper=20)
save_table(basket_stats.groupby('basket_items_clipped', observed=True).agg(
    baskets=('customer_key', 'size'), avg_value=('basket_value', 'mean'), avg_unique_articles=('unique_articles', 'mean')
).reset_index(), 'basket_size_summary')

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
sns.countplot(data=basket_stats, x='basket_items_clipped', hue='basket_items_clipped', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Shopping basket item distribution')
axes[0].set_xlabel('Items per basket (20+)')
axes[0].set_ylabel('Baskets')
axes[0].yaxis.set_major_formatter(FuncFormatter(comma_formatter))
sns.histplot(basket_stats['basket_value'], bins=60, ax=axes[1], color='#e6550d')
axes[1].set_title('Basket value distribution')
axes[1].set_xlabel('Basket value')
axes[1].set_ylabel('Baskets')
sns.scatterplot(data=basket_stats.sample(min(300_000, len(basket_stats)), random_state=2026), x='basket_items', y='basket_value', s=7, alpha=0.18, ax=axes[2], color='#31a354')
axes[2].set_title('Basket items vs basket value')
axes[2].set_xlabel('Items per basket')
axes[2].set_ylabel('Basket value')
fig.tight_layout()
save_figure(fig, '11_basket_analysis')

active_dates = transactions[['customer_key', 't_dat']].drop_duplicates().sort_values(['customer_key', 't_dat'])
active_dates['visit_gap_days'] = active_dates.groupby('customer_key')['t_dat'].diff().dt.days
visit_gaps = active_dates['visit_gap_days'].dropna()
pair_events = transactions[['customer_key', 'article_num', 't_dat']].drop_duplicates().sort_values(['customer_key', 'article_num', 't_dat'])
pair_events['same_item_gap_days'] = pair_events.groupby(['customer_key', 'article_num'])['t_dat'].diff().dt.days
same_item_gaps = pair_events['same_item_gap_days'].dropna()
pair_stats = pair_events.groupby(['customer_key', 'article_num'], as_index=False).agg(purchase_times=('t_dat', 'size'))
repeat_pair_count = int((pair_stats['purchase_times'] > 1).sum())
repeat_pair_share = repeat_pair_count / len(pair_stats)
repeat_event_share = float((pair_stats['purchase_times'] - 1).sum() / pair_stats['purchase_times'].sum())

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
sns.histplot(visit_gaps[visit_gaps <= 180], bins=60, ax=axes[0], color='#08519c')
axes[0].set_title('Customer active-day gap distribution')
axes[0].set_xlabel('Days between active days (<=180)')
axes[0].set_ylabel('Gaps')
sns.histplot(same_item_gaps[same_item_gaps <= 365], bins=60, ax=axes[1], color='#756bb1')
axes[1].set_title('Same-item repurchase gap distribution')
axes[1].set_xlabel('Days between same-item purchases (<=365)')
axes[1].set_ylabel('Gaps')
gap_frame = pd.DataFrame({'Gap type': ['Active-day gap', 'Same-item gap'], 'Median days': [visit_gaps.median(), same_item_gaps.median()]})
sns.barplot(data=gap_frame, x='Gap type', y='Median days', hue='Gap type', ax=axes[2], palette='Set2', legend=False)
axes[2].set_title('Median behavior gap')
axes[2].set_xlabel('')
axes[2].set_ylabel('Median days')
fig.tight_layout()
save_figure(fig, '12_purchase_gap_analysis')

behavior_summary = {
    'customers_with_transactions': int(len(customer_stats)),
    'median_lifetime_purchases': float(customer_stats['purchases'].median()),
    'mean_lifetime_purchases': float(customer_stats['purchases'].mean()),
    'median_active_days': float(customer_stats['active_days'].median()),
    'median_recency_days': float(customer_stats['recency_days'].median()),
    'median_basket_items': float(basket_stats['basket_items'].median()),
    'mean_basket_items': float(basket_stats['basket_items'].mean()),
    'median_visit_gap_days': float(visit_gaps.median()),
    'median_same_item_gap_days': float(same_item_gaps.median()),
    'repeat_customer_article_pair_share': float(repeat_pair_share),
    'repeat_event_share': float(repeat_event_share),
    'elapsed_seconds': round(time.time() - behavior_started, 3),
}
print(json.dumps(behavior_summary, ensure_ascii=False, indent=2))

## 4. 用户画像分析

将交易行为与年龄、会员状态、新闻订阅状态交叉分析，评估哪些画像字段对排序模型有价值，并为分群热门冷启动召回确定分组方式。

In [ ]:
profile_started = time.time()
profile_base = customers[['customer_key', 'age', 'club_member_status', 'fashion_news_frequency', 'fn_flag', 'active_flag']].merge(customer_stats, on='customer_key', how='left')
profile_base['has_transaction'] = profile_base['purchases'].notna()
profile_base['purchases_filled'] = profile_base['purchases'].fillna(0)
profile_base['revenue_filled'] = profile_base['revenue'].fillna(0)
profile_base['age_group'] = pd.cut(
    profile_base['age'], bins=[15, 25, 35, 45, 55, 65, 100],
    labels=['16-25', '26-35', '36-45', '46-55', '56-65', '66+']
)
profile_base['age_group'] = profile_base['age_group'].cat.add_categories('Unknown').fillna('Unknown')

age_summary = profile_base.groupby('age_group', observed=False).agg(
    customers=('customer_key', 'size'), active_share=('has_transaction', 'mean'),
    avg_purchases=('purchases_filled', 'mean'), median_purchases=('purchases_filled', 'median'),
    avg_revenue=('revenue_filled', 'mean')
).reset_index()
save_table(age_summary, 'age_group_summary')
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
sns.barplot(data=age_summary, x='age_group', y='customers', hue='age_group', ax=axes[0], palette='Set2', legend=False)
axes[0].set_title('Customer count by age group')
axes[0].set_xlabel('Age group')
axes[0].set_ylabel('Customers')
axes[0].tick_params(axis='x', rotation=20)
sns.barplot(data=age_summary, x='age_group', y='avg_purchases', hue='age_group', ax=axes[1], palette='Set2', legend=False)
axes[1].set_title('Average lifetime purchases by age')
axes[1].set_xlabel('Age group')
axes[1].set_ylabel('Average purchases')
axes[1].tick_params(axis='x', rotation=20)
sns.barplot(data=age_summary, x='age_group', y='active_share', hue='age_group', ax=axes[2], palette='Set2', legend=False)
axes[2].set_title('Transaction coverage by age group')
axes[2].set_xlabel('Age group')
axes[2].set_ylabel('Customers with transactions')
axes[2].tick_params(axis='x', rotation=20)
fig.tight_layout()
save_figure(fig, '13_age_profile_analysis')

def profile_group_summary(column):
    summary = profile_base.groupby(column, observed=False).agg(
        customers=('customer_key', 'size'), active_share=('has_transaction', 'mean'),
        avg_purchases=('purchases_filled', 'mean'), avg_revenue=('revenue_filled', 'mean')
    ).sort_values('customers', ascending=False).reset_index()
    summary['customer_share'] = summary['customers'] / summary['customers'].sum()
    return summary

member_summary = profile_group_summary('club_member_status')
news_summary = profile_group_summary('fashion_news_frequency')
save_table(member_summary, 'club_member_summary')
save_table(news_summary, 'fashion_news_summary')
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
sns.barplot(data=member_summary, x='club_member_status', y='customer_share', hue='club_member_status', ax=axes[0, 0], palette='Set2', legend=False)
axes[0, 0].set_title('Club member status distribution')
axes[0, 0].set_xlabel('')
axes[0, 0].set_ylabel('Customer share')
axes[0, 0].tick_params(axis='x', rotation=20)
sns.barplot(data=member_summary, x='club_member_status', y='avg_purchases', hue='club_member_status', ax=axes[0, 1], palette='Set2', legend=False)
axes[0, 1].set_title('Average purchases by member status')
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Average purchases')
axes[0, 1].tick_params(axis='x', rotation=20)
sns.barplot(data=news_summary, x='fashion_news_frequency', y='customer_share', hue='fashion_news_frequency', ax=axes[1, 0], palette='Set2', legend=False)
axes[1, 0].set_title('Fashion news frequency distribution')
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('Customer share')
axes[1, 0].tick_params(axis='x', rotation=20)
sns.barplot(data=news_summary, x='fashion_news_frequency', y='avg_purchases', hue='fashion_news_frequency', ax=axes[1, 1], palette='Set2', legend=False)
axes[1, 1].set_title('Average purchases by news frequency')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('Average purchases')
axes[1, 1].tick_params(axis='x', rotation=20)
fig.tight_layout()
save_figure(fig, '14_member_and_news_profile')

profile_summary = {
    'unknown_age_share': float(profile_base['age'].isna().mean()),
    'unknown_club_status_share': float((profile_base['club_member_status'] == 'UNKNOWN').mean()),
    'unknown_news_frequency_share': float((profile_base['fashion_news_frequency'] == 'UNKNOWN').mean()),
    'highest_avg_purchase_age_group': str(age_summary.loc[age_summary['avg_purchases'].idxmax(), 'age_group']),
    'highest_avg_purchase_member_status': str(member_summary.loc[member_summary['avg_purchases'].idxmax(), 'club_member_status']),
    'elapsed_seconds': round(time.time() - profile_started, 3),
}
print(json.dumps(profile_summary, ensure_ascii=False, indent=2))

## 5. 近期热门稳定性与冷启动覆盖

使用数据最后 7 天作为测试期，比较不同历史窗口 Top 商品与用户的覆盖率。该分析不用于最终评分，只用于直观评估热门召回、复购召回和冷启动策略的适用性。

In [ ]:
window_started = time.time()
max_date = transactions['t_dat'].max()
target_start = max_date - pd.Timedelta(days=6)
target = transactions[transactions['t_dat'] >= target_start]
target_article_set = set(target['article_num'].unique())
target_customer_set = set(target['customer_key'].unique())
submission_key_set = set(submission['customer_key'].unique())
window_days_list = [7, 14, 28, 91, 364]
window_rows = []
latest_top100 = set(transactions[transactions['t_dat'] >= max_date - pd.Timedelta(days=6)]['article_num'].value_counts().head(100).index)
for window_days in window_days_list:
    history = transactions[transactions['t_dat'] < target_start]
    window = history[history['t_dat'] >= target_start - pd.Timedelta(days=window_days)]
    article_counts = window['article_num'].value_counts()
    top100 = set(article_counts.head(100).index)
    top12 = set(article_counts.head(12).index)
    target_purchase_hits = target['article_num'].isin(top12)
    active_customers = set(window['customer_key'].unique())
    window_rows.append({
        'window_days': window_days,
        'purchases': int(len(window)),
        'active_customers': int(len(active_customers)),
        'submission_customer_coverage': len(active_customers & submission_key_set) / len(submission_key_set),
        'target_customer_coverage': len(active_customers & target_customer_set) / len(target_customer_set),
        'top100_overlap_with_latest7': len(top100 & latest_top100) / 100,
        'top100_jaccard_with_latest7': len(top100 & latest_top100) / len(top100 | latest_top100),
        'final_week_top12_purchase_coverage': float(target_purchase_hits.mean()),
    })
window_summary = pd.DataFrame(window_rows)
save_table(window_summary, 'window_popularity_and_coverage')

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
sns.lineplot(data=window_summary, x='window_days', y='submission_customer_coverage', marker='o', ax=axes[0, 0], color='#08519c')
axes[0, 0].set_title('Historical window customer coverage')
axes[0, 0].set_xlabel('History window days')
axes[0, 0].set_ylabel('Submission customer coverage')
sns.lineplot(data=window_summary, x='window_days', y='target_customer_coverage', marker='o', ax=axes[0, 1], color='#31a354')
axes[0, 1].set_title('Coverage of final-week active customers')
axes[0, 1].set_xlabel('History window days')
axes[0, 1].set_ylabel('Final-week active coverage')
sns.lineplot(data=window_summary, x='window_days', y='top100_overlap_with_latest7', marker='o', ax=axes[1, 0], color='#756bb1')
axes[1, 0].set_title('Top-100 article stability vs latest week')
axes[1, 0].set_xlabel('History window days')
axes[1, 0].set_ylabel('Overlap ratio')
sns.lineplot(data=window_summary, x='window_days', y='final_week_top12_purchase_coverage', marker='o', ax=axes[1, 1], color='#e6550d')
axes[1, 1].set_title('Global Top-12 coverage of final-week purchases')
axes[1, 1].set_xlabel('History window days')
axes[1, 1].set_ylabel('Purchase coverage')
for axis in axes.flat:
    axis.set_xscale('log')
fig.tight_layout()
save_figure(fig, '15_window_coverage_stability')

recency_bins = [-1, 7, 30, 90, 180, 365, 10_000]
recency_labels = ['0-7d', '8-30d', '31-90d', '91-180d', '181-365d', '365d+']
customer_stats['recency_segment'] = pd.cut(customer_stats['recency_days'], bins=recency_bins, labels=recency_labels)
recency_summary = customer_stats.groupby('recency_segment', observed=False).agg(
    customers=('customer_key', 'size'), avg_purchases=('purchases', 'mean'), avg_revenue=('revenue', 'mean')
).reset_index()
recency_summary['customer_share'] = recency_summary['customers'] / recency_summary['customers'].sum()
save_table(recency_summary, 'customer_recency_summary')
fig, axis = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=recency_summary, x='recency_segment', y='customer_share', hue='recency_segment', palette='Set2', ax=axis, legend=False)
axis.set_title('Transaction customers by recency segment')
axis.set_xlabel('Days since last purchase')
axis.set_ylabel('Customer share')
fig.tight_layout()
save_figure(fig, '16_customer_recency_distribution')

target_customer_profile = customer_stats[customer_stats['customer_key'].isin(target_customer_set)].copy()
history_before_target = transactions[transactions['t_dat'] < target_start]
returning_mask = target_customer_profile['customer_key'].isin(history_before_target['customer_key'].unique())
window_summary_dict = {
    'target_start': str(target_start.date()),
    'target_end': str(max_date.date()),
    'target_purchases': int(len(target)),
    'target_customers': int(len(target_customer_set)),
    'target_articles': int(len(target_article_set)),
    'submission_customers_active_in_target': float(len(target_customer_set & submission_key_set) / len(submission_key_set)),
    'target_customers_with_prior_history': float(returning_mask.mean()),
    'no_transaction_customer_share': float(1 - customers['customer_key'].isin(customer_stats['customer_key']).mean()),
    'best_top12_window_by_purchase_coverage': int(window_summary.loc[window_summary['final_week_top12_purchase_coverage'].idxmax(), 'window_days']),
    'elapsed_seconds': round(time.time() - window_started, 3),
}
print(json.dumps(window_summary_dict, ensure_ascii=False, indent=2))

In [ ]:
analysis_summary = {
    'generated_at': time.strftime('%Y-%m-%d %H:%M:%S %z'),
    'data_source': str(CLEAN_DIR) if CLEAN_DIR else f'raw fallback: {RAW_DIR}',
    'transactions': int(len(transactions)),
    'customers': int(len(customers)),
    'articles': int(len(articles)),
    'submission_customers': int(len(submission)),
    'temporal': temporal_summary,
    'item': item_summary,
    'behavior': behavior_summary,
    'profile': profile_summary,
    'window': window_summary_dict,
    'outputs': {
        'figures': sorted(path.name for path in FIGURE_DIR.glob('*.png')),
        'tables': sorted(path.name for path in TABLE_DIR.glob('*.csv')),
    },
}
write_path = EDA_DIR / 'eda_summary.json'
write_path.write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(analysis_summary, ensure_ascii=False, indent=2))

## 核心分析结论

1. **时间强季节性**：每日交易量和销售额存在明显波动与季节峰值，验证周、月、节假日和促销窗口特征的重要性。
2. **商品高度长尾**：极少数热门商品贡献了显著购买量，但 Top 100 无法覆盖全部需求，需要全局热门、分群热门与个性化召回互补。
3. **复购信号重要**：大量用户存在同商品或同偏好重复购买，时间衰减复购召回应作为核心通道。
4. **购物篮共现有价值**：购物篮大小分布显示多数篮内商品数较少，但仍有足量多商品篮可支持同天/同周共现挖掘。
5. **画像存在区分度**：年龄、会员状态与新闻订阅频率对应的购买强度不同，适合构建分群热门与交叉特征。
6. **冷启动不可忽略**：提交顾客中仅一部分在最后 7 天有交易，且不少顾客缺少近期历史，因此必须设计分层兜底策略。